# Update exposure data with user-input data

This notebook demonstrates how to update the exposure data in an existing Delft-FIAT model with user-input data. Examples will be shown about how to update the ground_flht and the max potential damages (this will be added later!).

## **Step 0**: Import required packages
First we need to import the necessary python packages.

In [1]:
from hydromt_fiat.fiat import FiatModel
from hydromt.log import setuplog
from pathlib import Path
import shutil
import geopandas as gpd
from hydromt.gis_utils import utm_crs

## **Step 1**: Configure
The first step is to set up the configuration needed to initialize the FIAT model. Begin by specifying where the model that should be updated is saved, locating the required data catalog, and setting up the logger.

In [ ]:
model_folder = Path().absolute() / "data" / "update_ground_floor_height" / "fiat_model"
data_catalog = Path().absolute() / "data" / "hydromt_fiat_catalog_USA.yml"
logger = setuplog("hydromt_fiat", log_level=10)

## **Step 2**: Initialize and read existing model
In this step we initialize HydroMT-FIAT in read-mode (`mode="r"`) with the `model_folder`, `data_catalog`, and `logger` that we defined above. We also read in the existing dummy model.

In [ ]:
fm = FiatModel(root=model_folder, mode="r", data_libs=[data_catalog], logger=logger)
fm.read()

## **Step 3**: Update the ground_flht of the exposure data
Next, we will update the ground_flht of some of the exposure assets in our model. We have created some ficticious data that represents elevation certificates. We will first inspect the current exposure data and ground_flhts and the "elevation certificate" data.

As can be seen below, apparently the ground_flht of all assets is set to 1 (foot).

In [ ]:
original_exposure = fm.exposure.get_full_gdf(fm.exposure.exposure_db)
original_exposure["ground_flht"].unique()

Now we will load the "elevation certificate" data and visualize it together with the exposure assets below on the map.

In [ ]:
elevation_certificates_path = (
    Path().absolute()
    / "data"
    / "update_ground_floor_height"
    / "fake_elevation_certificates.gpkg"
)
elevation_certificates = gpd.read_file(elevation_certificates_path)

# Plot the exposure and elevation certificates data
m = elevation_certificates.explore(column="FFE", cmap="Reds", style_kwds={"radius": 5})
m = original_exposure.explore(m=m, column="ground_flht", cmap="Reds")
m

We will use a maximum distance of 10 meter to join the elevation certificate data to their nearest building. The attribute in the elevation certificate data that we need to use is `FFE`, which stands for First Floor Elevation and is often used in the US to indicate the ground_flht with.

In [ ]:
attribute = "FFE"
method = "nearest"
max_dist = 50

fm.exposure.setup_ground_floor_height(
    ground_floor_height=elevation_certificates_path,
    gfh_attribute_name=attribute,
    gfh_method=method,
    max_dist=max_dist,
)

## **Step 4**: Check the resulting exposure data
As you can see below, the ground_flht of the exposure assets also contains other values.

In [ ]:
updated_exposure = fm.exposure.get_full_gdf(fm.exposure.exposure_db)
updated_exposure["ground_flht"].unique()

We will make a buffer of 10 meter around the 

In [ ]:
# Check if the Ground Floor Heigh attribute is set correctly
nearest_utm = utm_crs(elevation_certificates.total_bounds)
elevation_certificates_projected = elevation_certificates.to_crs(nearest_utm)
elevation_certificates_projected.geometry = (
    elevation_certificates_projected.geometry.buffer(max_dist)
)

# Plot the exposure and elevation certificates data
m = elevation_certificates_projected.explore(column="FFE", cmap="Reds")
m = updated_exposure.explore(m=m, column="ground_flht", cmap="Reds")
m

## **Step 5**: Save the updated FIAT model
Set the folder where you want to save the model to and write it to that folder.

In [ ]:
save_folder = (
    Path().absolute() / "data" / "update_ground_floor_height" / "updated_fiat_model"
)

# Remove the new root folder if it already exists
if save_folder.exists():
    shutil.rmtree(save_folder)

# Set the new root and write the model
fm.set_root(save_folder)
fm.write()